# Thesis-insilico — Tier 1-3 on Google Colab (crash-resumable)

Runs the As(III)/m6A in silico pipeline end to end on Colab:

- **Tier 1** — Cys reactivity landscape (`03_cys_landscape.py`)
- **Tier 2** — pocket detection + non-covalent/covalent docking + ranking (`04`-`07`)
- **Tier 3** — molecular dynamics on the top hits (`scripts/md/`, GPU runtime required)

**Why this notebook is safe to re-run after a crash.** Colab sessions
disconnect, hit idle timeouts, or get reclaimed mid-run — that's normal, not
an error. Every stage in this pipeline checkpoints its progress to
`results/.checkpoints/` (per protein, or per docking box for Vina) and the
GROMACS production runs checkpoint every few minutes via `-cpi`. Those
checkpoints — plus all downloaded structures and results — live on **your
Google Drive**, not on the ephemeral Colab VM disk. So the fix for *any*
crash, at *any* stage, is always the same:

> **Reconnect the runtime and re-run the cells from the top.** Already-finished
> work is detected and skipped; only the remaining work resumes.

The only things that are *not* persisted are the installed software
(conda env, GROMACS) — those live on the local VM disk and get reinstalled
each fresh session, which is intentional (fast, and avoids the well-known
slowness of running conda off a Drive mount).

## One-time setup before you start

1. `Runtime → Change runtime type`: for Tier 1-2 a standard CPU runtime is
   enough; for **Tier 3 you need a GPU** (T4 is fine — pick "T4 GPU").
   You can run Tier 1-2 first on CPU, then switch to GPU only for Tier 3
   (switching runtime type restarts the VM, which is fine — your data is
   on Drive).
2. Have a Google Drive with a few GB free — that's where all pipeline
   outputs and checkpoints will live, under `MyDrive/Thesis-insilico-run/`.


In [ ]:
# Sanity-check the runtime: GPU (for Tier 3), CPU count, disk.
import subprocess
print(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip(), "vCPUs")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU:", gpu.stdout.strip())
else:
    print("No GPU visible — fine for Tier 1-2. Switch to a GPU runtime before Tier 3.")


## 1. Mount Google Drive — this is where all progress persists

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Thesis-insilico-run"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Persistent run directory:", DRIVE_ROOT)


## 2. Configure and clone the repo

`REPO_DIR` is on the local (ephemeral) Colab disk — code is cheap to
re-clone every session. Change `BRANCH` if you're running a different
branch than the one this notebook shipped on.


In [ ]:
import os

REPO_URL = "https://github.com/Wachirawut2023/Thesis-insilico.git"
BRANCH = "claude/tier-1-3-google-collab-xiy0w4"
REPO_DIR = "/content/Thesis-insilico"

os.environ["REPO_URL"] = REPO_URL
os.environ["BRANCH"] = BRANCH
os.environ["REPO_DIR"] = REPO_DIR
os.environ["DRIVE_ROOT"] = DRIVE_ROOT


In [ ]:
%%bash
set -euo pipefail
# /content is wiped every fresh session, so always start with a clean clone.
rm -rf "$REPO_DIR"
git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
cd "$REPO_DIR" && git log -1 --oneline


## 3. Link persistent data/results directories onto Drive

`data/pdb`, `data/af`, `data/prepared`, and all of `results/` (including
`results/.checkpoints/` and `results/md/`) get symlinked into your Drive
folder. Everything the pipeline computes — and every checkpoint marker —
now survives a crash, a disconnect, or even switching the runtime type.
Re-running this cell is always safe (it's idempotent).


In [ ]:
%%bash
set -euo pipefail
mkdir -p "$DRIVE_ROOT"/data/pdb "$DRIVE_ROOT"/data/af "$DRIVE_ROOT"/data/prepared "$DRIVE_ROOT"/results
cd "$REPO_DIR"
for d in data/pdb data/af data/prepared results; do
  if [ -L "$d" ]; then
    continue
  fi
  rm -rf "$d"
  ln -s "$DRIVE_ROOT/$d" "$d"
done
ls -la data/ results/
echo "OK: data/{pdb,af,prepared} and results/ now point at Drive."


## 4. Install Tier 1-2 dependencies (CPU stages)

Installs Miniforge + the conda env from `env/environment.yml` (everything
except GROMACS, which Tier 3 installs separately below). This runs fresh
each session — it takes a few minutes, but it's the local VM disk, not
Drive, so there's no benefit to persisting it and real cost to trying
(conda envs on a Drive FUSE mount are very slow).


In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/miniforge3 ]; then
  curl -fsSL -o /tmp/miniforge.sh \
    "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
  bash /tmp/miniforge.sh -b -p /content/miniforge3
fi
source /content/miniforge3/etc/profile.d/conda.sh

if ! conda env list | grep -q '^arsenic-m6a '; then
  mamba env create -f "$REPO_DIR/env/environment.yml"
else
  echo "env arsenic-m6a already exists this session"
fi
conda activate arsenic-m6a
python -c "import Bio, MDAnalysis, freesasa; print('env OK')"


## 5. Prepare the As(OH)3 ligand (fast, regenerated fresh each session)

In [ ]:
%%bash
set -euo pipefail
source /content/miniforge3/etc/profile.d/conda.sh && conda activate arsenic-m6a
cd "$REPO_DIR"
bash ligands/prepare_as3.sh


## 6. Tier 1 — fetch structures, verify numbering, prepare receptors, Cys landscape

`fetch`/`prep` already skip files they've already downloaded/prepared.
`cys` checkpoints per protein in `results/.checkpoints/03_cys.done`. If this
cell is interrupted partway through, just re-run it — completed proteins
are skipped, not redone.

Optional: set `SMOKE_GENES` (e.g. `"METTL3,ALKBH5,YTHDF2"`) below to run a
3-target smoke test before committing to the full ~18-target panel.


In [ ]:
import os
SMOKE_GENES = ""  # e.g. "METTL3,ALKBH5,YTHDF2" for a quick smoke test; "" for the full panel
os.environ["SMOKE_GENES"] = SMOKE_GENES


In [ ]:
%%bash
set -euo pipefail
source /content/miniforge3/etc/profile.d/conda.sh && conda activate arsenic-m6a
cd "$REPO_DIR"
python scripts/01_fetch_structures.py
python scripts/00_verify_mettl3_cys.py
python scripts/02_prepare_receptors.py
python scripts/03_cys_landscape.py


## 7. Tier 2 — pocket detection, docking, ranking

This is the slowest CPU stage — `fpocket` runs per protein (up to 10 min
timeout each) and Vina runs per docking box (up to 30 min timeout each,
and a protein commonly has 2-4 boxes). Both stages checkpoint as they go
(`04_pockets` per gene, `05_dock_nc` per gene+box in
`results/.checkpoints/`), appending finished rows straight to the TSV
outputs. **If this cell disconnects, just re-run it** — it picks up at the
next un-checkpointed protein/box instead of starting over.


In [ ]:
%%bash
set -euo pipefail
source /content/miniforge3/etc/profile.d/conda.sh && conda activate arsenic-m6a
cd "$REPO_DIR"
python scripts/04_pocket_detection.py
python scripts/05_dock_noncovalent.py
python scripts/06_dock_covalent.py
python scripts/07_score_and_rank.py
python scripts/08_visualize_top_hits.py


In [ ]:
%%bash
# Quick look at Tier 1-2 results
cd "$REPO_DIR"
echo "--- composite_ranking.tsv (top 10) ---"
column -t -s $'\t' results/composite_ranking.tsv | head -11


## 8. Tier 3 — Molecular dynamics on the top hits (GPU required)

**Switch the runtime to a GPU now if you haven't** (`Runtime → Change
runtime type → T4 GPU`). Changing runtime type restarts the VM — that's
fine, just re-run cells 1-4 above (Drive mount, clone, symlink, conda env)
after reconnecting; your Tier 1-2 results are already safe on Drive.

Each protein/mode's 50 ns production run checkpoints via GROMACS'
`-cpi md.cpt -append` every few minutes (`MD_CPT_MIN`, default 5). A run
interrupted mid-production resumes from its last checkpoint rather than
restarting the trajectory. `run_all.sh` itself skips any protein/mode
whose `md.gro` already exists. So however this cell dies — disconnect,
crash, 12h Colab cap — **re-running it continues exactly where it stopped.**

Optional filters (set before running):
- `GENES_FILTER="TXN1,PIN1"` — restrict to a subset of proteins.
- `MODES_FILTER="apo"` — apo-only or bound-only.
- `MD_MAXH="5"` — make each protein's production step stop cleanly (and
  checkpoint) after 5 wall-clock hours instead of running until Colab
  kills it. Optional — GROMACS' own periodic checkpoint means an abrupt
  kill is safe too, but a clean stop is tidier and never corrupts a
  checkpoint.


In [ ]:
%%bash
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi >/dev/null 2>&1; then
  echo "No GPU detected. Runtime -> Change runtime type -> GPU (T4), then reconnect and re-run from cell 1."
  exit 1
fi
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
%%bash
# Note: deliberately NOT set -u — the conda-forge GROMACS package's
# activate.d hook sources GMXRC.bash, which references variables
# (GMXLDLIB etc.) that are unset on first activation. set -e + pipefail
# is enough to catch real failures without tripping on that.
set -eo pipefail
source /content/miniforge3/etc/profile.d/conda.sh

# conda-forge's plain "nompi" gromacs build offloads to GPU via OpenCL --
# and GROMACS' OpenCL backend does not support Volta/Turing/Ampere-or-newer
# NVIDIA GPUs (the T4 included) for compute. gmx enumerates the card, marks
# it "incompatible", and silently drops to CPU-only -- which is exactly
# what was happening here. conda-forge also publishes a real CUDA-enabled
# build (build string "nompi_cuda*"); pin it explicitly, since the solver
# won't prefer it on its own (the plain build doesn't depend on
# cuda-version, so it's just as solver-valid and commonly gets picked
# first).
GMX_SPEC="gromacs=2024.5=nompi_cuda*"

# Make conda's CUDA-capability detection deterministic instead of relying
# on it correctly probing the driver at solve time.
CUDA_VER=$(nvidia-smi 2>/dev/null | grep -oP 'CUDA Version:\s*\K[0-9]+\.[0-9]+' | head -1)
export CONDA_OVERRIDE_CUDA="${CUDA_VER:-12.4}"
echo "CONDA_OVERRIDE_CUDA=$CONDA_OVERRIDE_CUDA"

is_cuda_build() {
  conda activate md 2>/dev/null || return 1
  command -v gmx >/dev/null 2>&1 || return 1
  gmx --version 2>&1 | grep -qi "GPU support:.*CUDA"
}

if conda env list | grep -q '^md '; then
  if is_cuda_build; then
    echo "env md already exists and is CUDA-enabled — making sure pdbfixer is present"
    mamba install -n md -c conda-forge -y pdbfixer
  else
    echo "env md exists but isn't CUDA-enabled (leftover from an earlier attempt) — recreating"
    conda deactivate 2>/dev/null || true
    conda env remove -n md -y
  fi
fi
if ! conda env list | grep -q '^md '; then
  mamba create -n md -c conda-forge -c bioconda -y \
    "$GMX_SPEC" python=3.11 mdanalysis freesasa biopython pdbfixer pandas numpy matplotlib
fi

if ! is_cuda_build; then
  echo "ERROR: gromacs env is not CUDA-enabled after install/recreate."
  gmx --version 2>&1 | grep -i "GPU support" || echo "  (gmx not even on PATH)"
  echo "This means conda-forge has no '$GMX_SPEC' build resolvable right now"
  echo "(e.g. a channel hiccup, or 2024.5 aged out of the channel -- check"
  echo "what's current at https://anaconda.org/conda-forge/gromacs/files and"
  echo "bump GMX_SPEC above)."
  echo "As a last resort, set MD_GPU=0 in the run cell below for a much"
  echo "slower CPU-only run."
  exit 1
fi
echo "Confirmed CUDA build:"
gmx --version 2>&1 | grep -i "GPU support"
python -c "import pdbfixer" && echo "pdbfixer OK"


### Why the GPU wasn't being used before

The default conda-forge `gromacs` install resolves to a build that offloads
to GPU via **OpenCL** — and GROMACS' OpenCL backend doesn't support
Volta/Turing/Ampere-or-newer NVIDIA GPUs (the T4 included) for compute. gmx
detects the card fine, reports it "incompatible", and silently runs
CPU-only — no error, just a much slower run that looks like the GPU was
never used. The cell above instead pins the **CUDA-enabled build**
(`nompi_cuda*`) and fails loudly if that build can't be resolved, instead
of quietly falling back.

If it does fail, read the error message it prints — likely fixes are
bumping the pinned GROMACS version to whatever's current on
[anaconda.org/conda-forge/gromacs/files](https://anaconda.org/conda-forge/gromacs/files),
or as a stopgap: run Tier 3 CPU-only (`MD_GPU=0` below, very slow) or use
`infra/gcp-gpu/README.md` / `infra/runpod/README.md` for a full GPU VM.


In [ ]:
import os
GENES_FILTER = ""   # e.g. "TXN1,PIN1" to restrict; "" = all top-8 hits from md_targets.tsv
MODES_FILTER = ""   # "apo", "bound", or "" for both
MD_MAXH = "0"        # e.g. "5" to force a clean stop after 5h; "0" = run until finished or killed
MD_GPU = "1"         # "0" to force CPU-only (only if the CUDA-build install above failed and you're not able to fix it)

os.environ["GENES_FILTER"] = GENES_FILTER
os.environ["MODES_FILTER"] = MODES_FILTER
os.environ["MD_MAXH"] = MD_MAXH
os.environ["MD_GPU"] = MD_GPU


In [ ]:
%%bash
# set -u deliberately omitted here too — see the GMXRC note in the env-install
# cell above; conda activate md sources it as part of activation.
set -eo pipefail
source /content/miniforge3/etc/profile.d/conda.sh
conda activate md || true
command -v gmx >/dev/null || { echo "gmx not on PATH after activation — re-run the env-install cell above"; exit 1; }
cd "$REPO_DIR"
bash scripts/md/run_all.sh


### Monitoring a long Tier-3 run

Run this in a **separate cell** while `run_all.sh` is executing above (Colab
lets you run other cells while one is busy) to check progress without
waiting for the whole panel to finish:


In [ ]:
%%bash
cd "$REPO_DIR"
echo "--- GPU ---"
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader 2>/dev/null || echo "n/a"
echo "--- completed protein/mode runs ---"
ls results/md/*/md.gro 2>/dev/null | sed 's#results/md/##; s#/md.gro##' || echo "none finished yet"
echo "--- tail of orchestrator log ---"
tail -n 20 results/md/run_all.log 2>/dev/null || echo "no log yet"


## 9. Export results (optional)

Everything is already on Drive under
`MyDrive/Thesis-insilico-run/results/`, but this makes a single tarball
for easy download or sharing.


In [ ]:
%%bash
set -euo pipefail
# Stage a local copy before tarring. Reading directly off the Drive FUSE
# mount (/content/drive/...) can make tar report spurious "file changed
# as we read it" errors even when nothing is writing — a known Colab/Drive
# quirk, not a real race. Copying locally first avoids it and is faster.
STAGE=/content/export_results
rm -rf "$STAGE"
mkdir -p "$STAGE"
command -v rsync >/dev/null || apt-get -qq install -y rsync
rsync -a --exclude='md/*/*.trr' --exclude='md/*/*.xtc' "$DRIVE_ROOT/results/" "$STAGE/results/"

OUT="$DRIVE_ROOT/results_$(date +%Y%m%d_%H%M).tar.gz"
tar czf "$OUT" -C /content/export_results results
ls -lh "$OUT"


## Resetting a stage (if you want to force a full recompute)

Checkpoints live in `results/.checkpoints/<stage>.done` on Drive. To force
a specific stage to redo everything, delete its checkpoint file (and,
usually, its output TSV) and re-run that stage's cell — e.g.:

```python
%%bash
cd "$REPO_DIR"
rm -f results/.checkpoints/05_dock_nc.done results/docking_noncovalent.tsv
```

For Tier 3, deleting `results/md/<GENE>_<mode>/` removes that run's
checkpoint entirely and `run_all.sh` will redo it from scratch.
